# 03 — Grad-CAM Explainability Analysis

Visualize what the trained model "looks at" using Gradient-weighted Class Activation Mapping.

This helps validate that the model focuses on clinically relevant lesion features
rather than imaging artifacts (pen markings, device borders, etc.).

**Prerequisites:** Run `02_train_and_evaluate.ipynb` first to generate model checkpoints.

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Assumes you already ran notebook 02 which clones the repo and downloads data
    os.chdir('/content/advanced-topics')
    sys.path.insert(0, '/content/advanced-topics')
    DATA_DIR = '/content/data/HAM10000'
else:
    DATA_DIR = '../data/HAM10000'
    sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from src.config import DEVICE, MODELS_DIR, CLASS_NAMES, IMAGE_SIZE
from src.models import get_efficientnet_b0
from src.gradcam import GradCAM, get_target_layer, visualize_gradcam

print(f'Device: {DEVICE}')

In [ ]:
# --- Load Best Model ---
model = get_efficientnet_b0(num_classes=7, pretrained=False)
model.load_state_dict(torch.load(MODELS_DIR / 'efficientnet_b0_best.pth', map_location=DEVICE))
model = model.eval().to(DEVICE)
print('Model loaded successfully')

In [ ]:
# --- Select Representative Images (one per class) ---
metadata = pd.read_csv(Path(DATA_DIR) / 'HAM10000_metadata.csv')
image_dir = Path(DATA_DIR) / 'images'

# Pick one random image per class
samples = metadata.groupby('dx').apply(lambda x: x.sample(1, random_state=42)).reset_index(drop=True)
print(f'Selected {len(samples)} images (one per class):')
print(samples[['image_id', 'dx']].to_string(index=False))

In [ ]:
# --- Generate Grad-CAM for Each Class ---
label_map = {name: idx for idx, name in enumerate(sorted(metadata['dx'].unique()))}

fig, axes = plt.subplots(7, 3, figsize=(12, 28))

for row_idx, (_, sample) in enumerate(samples.iterrows()):
    img_path = image_dir / f"{sample['image_id']}.jpg"
    true_label = label_map[sample['dx']]
    
    # This generates the 3-panel visualization (original | heatmap | overlay)
    # For the grid, we do it manually:
    from src.gradcam import GradCAM, get_target_layer
    from torchvision import transforms
    from src.config import IMAGENET_MEAN, IMAGENET_STD
    
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    
    img = Image.open(img_path).convert('RGB')
    input_tensor = transform(img)
    
    target_layer = get_target_layer(model, 'efficientnet')
    gradcam = GradCAM(model, target_layer)
    heatmap, pred_class = gradcam.generate(input_tensor)
    
    heatmap_resized = np.array(Image.fromarray(
        (heatmap * 255).astype(np.uint8)
    ).resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)) / 255.0
    
    img_resized = np.array(img.resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0
    
    # Original
    axes[row_idx, 0].imshow(img_resized)
    axes[row_idx, 0].set_title(f"True: {sample['dx']}", fontsize=10)
    axes[row_idx, 0].axis('off')
    
    # Heatmap
    axes[row_idx, 1].imshow(heatmap_resized, cmap='jet')
    axes[row_idx, 1].set_title(f"Pred: {CLASS_NAMES[pred_class]}", fontsize=10)
    axes[row_idx, 1].axis('off')
    
    # Overlay
    overlay = img_resized * 0.5 + plt.cm.jet(heatmap_resized)[:, :, :3] * 0.5
    axes[row_idx, 2].imshow(overlay)
    correct = '✓' if pred_class == true_label else '✗'
    axes[row_idx, 2].set_title(f"Overlay {correct}", fontsize=10)
    axes[row_idx, 2].axis('off')

plt.suptitle('Grad-CAM Attention Maps — EfficientNet-B0', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../results/gradcam_all_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## Discussion

**What to look for in the Grad-CAM visualizations:**

- Does the model focus on the lesion itself (good) or on background/artifacts (bad)?
- For melanoma, does it attend to irregular borders and color variation?
- For vascular lesions, does it highlight the characteristic red/purple areas?
- Are there cases where the model focuses on pen markings or ruler edges (dataset bias)?

These insights help validate whether the model has learned clinically meaningful features
or is relying on spurious correlations in the training data.